In [1]:
from google.colab import drive
drive.mount('/content/drive')


%cd /content/drive/MyDrive/Project/Xet_Khen_Thuong/Fine-Tune-Qwen-Ner-Rel

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1rXa47xwluJ2JWUvIztf9x3d0HcbjwocA/Project/Xet_Khen_Thuong/Fine-Tune-Qwen-Ner-Rel


# Fine-tuning LLM Tiếng Việt bằng QLoRA/LoRA trên Google Colab để hỗ trợ tạo graph

---



---



Model: **Qwen2.5-1.5B-Instruct**  
Framework: Hugging Face Transformers + TRL + PEFT

> Trước khi chạy: vào **Runtime -> Change runtime type -> GPU** (chọn T4 hoặc cao hơn).

In [2]:
!pip install -q -U \
    transformers==4.57.6 \
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


Nếu Colab yêu cầu **restart runtime** sau khi cài, hãy chọn `Runtime -> Restart session` rồi chạy tiếp từ cell dưới.

In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("Không có GPU. Hãy chọn Runtime -> Change runtime type -> GPU")


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [4]:
import os
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    PeftModel
)

# Lưu ý: SFTConfig thay thế cho TrainingArguments trong các bản TRL mới.
# dataset_text_field / max_length / packing giờ là thuộc tính của SFTConfig,
# không còn truyền trực tiếp vào SFTTrainer nữa.
from trl import SFTTrainer, SFTConfig


## Dataset tiếng Việt mẫu

Bạn có thể thay thế danh sách `data` bên dưới bằng dữ liệu của riêng bạn (càng nhiều mẫu càng tốt, tối thiểu vài trăm mẫu để có kết quả ổn).

In [5]:
import json
import os

dataset_path = "DataTrain"
data_eval = "DataTest"

#Load dữ liệu từ thư mục
def load_data(dataset_path):
  dataset = []
  for filename in os.listdir(dataset_path):
      if filename.endswith('.json'):
          file_path = os.path.join(dataset_path, filename)

          with open(file_path, 'r', encoding='utf-8') as f:
              data = json.load(f)

          dataset.extend(data)
  return dataset

data = load_data(dataset_path=dataset_path)
data_eval = load_data(dataset_path=data_eval)

train_dataset = Dataset.from_list(data)
test_dataset = Dataset.from_list(data_eval)

print("Train:", len(train_dataset))
print("Test :", len(test_dataset))


Train: 306
Test : 23


In [6]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("Model:", model_name)
#module token
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Vocab size:", len(tokenizer))


Model: Qwen/Qwen2.5-1.5B-Instruct


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size: 151665


In [7]:
from src.preprocessing import format_example

SYSTEM_PROMPT =SYSTEM_PROMPT = """
Bạn là chuyên gia xử lý ngôn ngữ tự nhiên tiếng Việt và xây dựng Knowledge Graph từ văn bản pháp luật về thi đua, khen thưởng. Nhiệm vụ của bạn là phân tích câu đầu vào, trích xuất các thực thể và quan hệ được thể hiện trực tiếp trong câu.
**1. Nhãn thực thể hợp lệ (`entity_labels`):**
- `CA_NHAN`: Cá nhân.
- `TO_CHUC`: Tổ chức, doanh nghiệp.
- `CO_QUAN_NHA_NUOC`: Cơ quan nhà nước.
- `TAP_THE`: Tập thể.
- `DANH_HIEU_THI_DUA`: Danh hiệu thi đua hoặc danh hiệu được xét tặng.
- `THANH_TICH`: Thành tích, kết quả đạt được.
- `CAP_KHEN_THUONG`: Cấp có thẩm quyền khen thưởng hoặc cấp khen thưởng được đề cập.
- `DIEU_KIEN`: Điều kiện cần đáp ứng để được xét tặng.
- `VAN_BAN_PHAP_LUAT`: Văn bản pháp luật.
- `DIEU_KHOAN_PHAP_LUAT`: Điều, khoản, điểm hoặc quy định cụ thể được viện dẫn.
- `THOI_GIAN`: Mốc thời gian, năm hoặc giai đoạn.
- `THOI_HAN`: Thời hạn thực hiện hoặc thời hạn được quy định.
Chỉ sử dụng các nhãn trên. Không tự tạo nhãn mới. Chọn nhãn phản ánh chính xác nhất vai trò của thực thể trong câu.
**2. Nhãn quan hệ hợp lệ (`relation_labels`):**
- `AP_DUNG_CHO`: Danh hiệu, quy định hoặc chính sách áp dụng cho đối tượng được nêu rõ.
- `DAT_DANH_HIEU`: Chủ thể đạt hoặc được xác định là đã đạt một danh hiệu.
- `DAT_TIEU_CHUAN`: Chủ thể đáp ứng tiêu chuẩn được đề cập.
- `CO_THANH_TICH`: Chủ thể có thành tích được nêu trong câu.
- `CAN_CU_XET_TANG`: Thành tích, điều kiện hoặc căn cứ được nêu làm cơ sở xét tặng.
- `DUOC_QUY_DINH_TAI`: Nội dung được quy định tại văn bản hoặc điều khoản pháp luật được viện dẫn.
- `CO_SANG_KIEN`: Chủ thể có sáng kiến được đề cập.
- `CO_CONG_TRINH_KHOA_HOC`: Chủ thể có công trình khoa học được đề cập.
- `DUOC_XET_TRONG_THOI_GIAN`: Hoạt động xét tặng hoặc chủ thể được xét trong khoảng thời gian được nêu rõ.
- `DUOC_BINH_XET_THONG_QUA`: Việc bình xét được thực hiện hoặc thông qua theo nội dung được nêu rõ.
Chỉ sử dụng các quan hệ trên. Không tự tạo loại quan hệ mới. Không tạo quan hệ nếu câu không thể hiện đủ căn cứ ngữ nghĩa.
**3. Quy tắc trích xuất:**
- Trích xuất đầy đủ các thực thể quan trọng xuất hiện trực tiếp trong câu; giữ nguyên tên gọi, không làm mất thông tin pháp lý quan trọng.
- Mỗi thực thể có cấu trúc `{"text": "Tên thực thể", "label": "NHÃN_THỰC_THỂ"}`.
- Mỗi quan hệ có cấu trúc `{"subject": "Thực thể nguồn", "predicate": "NHÃN_QUAN_HỆ", "object": "Thực thể đích"}`.
- `subject` và `object` phải khớp chính xác với trường `text` của các thực thể trong `Entity`. Không tạo quan hệ tham chiếu đến thực thể chưa được trích xuất.
- Xác định đúng chiều quan hệ và vai trò của từng thực thể. Không đảo chiều quan hệ, không suy diễn quan hệ chỉ dựa trên kiến thức bên ngoài hoặc mối liên hệ thường gặp.
- Chỉ trích xuất thông tin được câu đề cập hoặc thể hiện rõ về mặt ngữ nghĩa. Không tự bổ sung đối tượng, thành tích, điều kiện, thời gian hay căn cứ pháp lý.
- Phân biệt danh hiệu thi đua với thành tích, điều kiện với tiêu chuẩn, văn bản pháp luật với điều khoản pháp luật; không gán nhãn chỉ dựa trên từ khóa riêng lẻ.
- Không tạo thực thể trùng lặp. Nếu một thực thể tham gia nhiều quan hệ được thể hiện rõ, hãy trích xuất đầy đủ các quan hệ đó.
- Nếu câu chỉ chứa thông tin mô tả hoặc không thể hiện quan hệ phù hợp với danh sách cho phép, vẫn trích xuất thực thể nhưng trả về `relations` rỗng. Không cố tạo quan hệ để làm đầy dữ liệu.
**4. Schema đầu ra bắt buộc:**
{
  "Entity": [
    {
      "text": "Tên thực thể",
      "label": "NHÃN_THỰC_THỂ"
    }
  ],
  "relations": [
    {
      "subject": "Tên thực thể nguồn",
      "predicate": "NHÃN_QUAN_HỆ",
      "object": "Tên thực thể đích"
    }
  ]
}
Chỉ trả về một đối tượng JSON hợp lệ theo schema trên. `Entity` và `relations` luôn phải tồn tại, kể cả khi một hoặc cả hai danh sách rỗng. Không thêm trường ngoài schema. Không sử dụng nhãn ngoài hai danh sách đã cung cấp. Không giải thích, không bình luận, không thêm Markdown hoặc bất kỳ văn bản nào ngoài JSON.
"""
train_dataset = train_dataset.map(
    format_example,
    fn_kwargs={
        "prompt": SYSTEM_PROMPT,
        "tokenizer": tokenizer,
    }
)
test_dataset = test_dataset.map(
    format_example,
    fn_kwargs={
        "prompt": SYSTEM_PROMPT,
        "tokenizer": tokenizer,
    }
)

print(train_dataset[0]["text"])


Map:   0%|          | 0/306 [00:00<?, ? examples/s]

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

<|im_start|>system

Bạn là chuyên gia xử lý ngôn ngữ tự nhiên tiếng Việt và xây dựng Knowledge Graph từ văn bản pháp luật về thi đua, khen thưởng. Nhiệm vụ của bạn là phân tích câu đầu vào, trích xuất các thực thể và quan hệ được thể hiện trực tiếp trong câu.
**1. Nhãn thực thể hợp lệ (`entity_labels`):**
- `CA_NHAN`: Cá nhân.
- `TO_CHUC`: Tổ chức, doanh nghiệp.
- `CO_QUAN_NHA_NUOC`: Cơ quan nhà nước.
- `TAP_THE`: Tập thể.
- `DANH_HIEU_THI_DUA`: Danh hiệu thi đua hoặc danh hiệu được xét tặng.
- `THANH_TICH`: Thành tích, kết quả đạt được.
- `CAP_KHEN_THUONG`: Cấp có thẩm quyền khen thưởng hoặc cấp khen thưởng được đề cập.
- `DIEU_KIEN`: Điều kiện cần đáp ứng để được xét tặng.
- `VAN_BAN_PHAP_LUAT`: Văn bản pháp luật.
- `DIEU_KHOAN_PHAP_LUAT`: Điều, khoản, điểm hoặc quy định cụ thể được viện dẫn.
- `THOI_GIAN`: Mốc thời gian, năm hoặc giai đoạn.
- `THOI_HAN`: Thời hạn thực hiện hoặc thời hạn được quy định.
Chỉ sử dụng các nhãn trên. Không tự tạo nhãn mới. Chọn nhãn phản ánh chính xác nhấ

In [8]:
import os
import sys

print("Thư mục hiện tại:", os.getcwd())
print("Các file trong thư mục hiện tại:", os.listdir())
print("Python path:", sys.path)

Thư mục hiện tại: /content/drive/.shortcut-targets-by-id/1rXa47xwluJ2JWUvIztf9x3d0HcbjwocA/Project/Xet_Khen_Thuong/Fine-Tune-Qwen-Ner-Rel
Các file trong thư mục hiện tại: ['DataTrain', 'DataTest', 'src', 'CheckPoints', 'Finetune_qwen_NER_REL.ipynb']
Python path: ['/content', '/env/python', '/usr/lib/python313.zip', '/usr/lib/python3.13', '/usr/lib/python3.13/lib-dynload', '', '/usr/local/lib/python3.13/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.13/dist-packages/IPython/extensions', '/root/.ipython']


In [9]:
import sys

project_root = (
    " /content/drive/MyDrive/Project/Xet_Khen_Thuong/Fine-Tune-Qwen-Ner-Rel"
)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [10]:
import os
import sys

print("Current directory:", os.getcwd())
print("\nsys.path:")
print("\n".join(sys.path))

print("\nTìm file config.py:")
for base in [
    "/content",
    "/content/drive/MyDrive/Project"
]:
    if os.path.exists(base):
        for root, dirs, files in os.walk(base):
            dirs[:] = [
                d for d in dirs
                if d not in {"__pycache__", ".git", ".venv"}
            ]

            if "config.py" in files:
                print("Tìm thấy:", os.path.join(root, "config.py"))

Current directory: /content/drive/.shortcut-targets-by-id/1rXa47xwluJ2JWUvIztf9x3d0HcbjwocA/Project/Xet_Khen_Thuong/Fine-Tune-Qwen-Ner-Rel

sys.path:
 /content/drive/MyDrive/Project/Xet_Khen_Thuong/Fine-Tune-Qwen-Ner-Rel
/content
/env/python
/usr/lib/python313.zip
/usr/lib/python3.13
/usr/lib/python3.13/lib-dynload

/usr/local/lib/python3.13/dist-packages
/usr/lib/python3/dist-packages
/usr/local/lib/python3.13/dist-packages/IPython/extensions
/root/.ipython

Tìm file config.py:
Tìm thấy: /content/drive/.shortcut-targets-by-id/1rXa47xwluJ2JWUvIztf9x3d0HcbjwocA/Project/Xet_Khen_Thuong/Fine-Tune-Qwen-Ner-Rel/src/config.py
Tìm thấy: /content/drive/MyDrive/Project/Xet_Khen_Thuong/Fine-Tune-Qwen-Ner-Rel/src/config.py


In [11]:
from pathlib import Path

src_dir = Path.cwd() / "src"

old_file = src_dir / "config (1).py"
new_file = src_dir / "config.py"

if old_file.exists():
    if new_file.exists():
        print("Đã có config.py, không ghi đè.")
    else:
        old_file.rename(new_file)
        print("Đã đổi tên thành config.py")
else:
    print("Không tìm thấy config (1).py")

Không tìm thấy config (1).py


## Cấu hình QLoRA 4-bit và load model

In [12]:
from src.config import  *

compute_dtype = torch.bfloat16

quantization_config = QuantizationConfig()
bnb_config = quantization_config.build()
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

print("Đã load model.")
print("Model dtype:", model.dtype)

from peft import prepare_model_for_kbit_training

# Bước chuẩn khi dùng QLoRA: đưa layernorm về fp32, bật input require_grad,
# chuẩn bị model tương thích với gradient checkpointing. Thiếu bước này
# cũng là nguyên nhân phổ biến gây lỗi dtype/NaN khi train.
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Đã load model.
Model dtype: torch.bfloat16


## Cấu hình training (`SFTConfig`)

**Lưu ý quan trọng:** trong các bản TRL mới (>=0.19–0.20 trở lên), `dataset_text_field`, `max_length` (đổi tên từ `max_seq_length`) và `packing` là thuộc tính của `SFTConfig`, không còn truyền trực tiếp vào `SFTTrainer` nữa. Đây là nguyên nhân phổ biến nhất gây lỗi `TypeError` khi chạy các đoạn code fine-tuning cũ.

In [13]:
lora_config = LoRAConfig()
peft_config = lora_config.build()
training_config = TrainingConfig()
training_args = training_config.build()


trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    args=training_args,

    # tokenizer= đã bị loại bỏ khỏi TRL, dùng processing_class thay thế
    processing_class=tokenizer,
)
trainer.model.print_trainable_parameters()


Adding EOS to train dataset:   0%|          | 0/306 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/306 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/306 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/306 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/306 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/23 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/23 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/23 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/23 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/23 [00:00<?, ? examples/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [14]:
# @title KIỂM TRA BF16 CÓ TƯƠNG THÍCH KHÔNG
import torch

print("PyTorch:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

# Kiểm tra phép nhân ma trận BF16 trực tiếp
try:
    a = torch.randn(128, 128, device="cuda", dtype=torch.bfloat16)
    b = torch.randn(128, 128, device="cuda", dtype=torch.bfloat16)

    c = a @ b
    print("BF16 GEMM thành công:", c.dtype)

except Exception as e:
    print("BF16 GEMM thất bại:", repr(e))

PyTorch: 2.11.0+cu130
CUDA version: 13.0
GPU: Tesla T4
Compute capability: (7, 5)
BF16 supported: True
BF16 GEMM thành công: torch.bfloat16


In [15]:
import trl.kernels.chunked_logprob as clp

# Buộc TRL dùng nhánh fallback tính toán FP32
clp._MM_OUT_DTYPE = False

print("_MM_OUT_DTYPE =", clp._MM_OUT_DTYPE)

_MM_OUT_DTYPE = False


## Bắt đầu fine-tuning

Với dataset mẫu và cấu hình trên, quá trình này chạy trong vài phút trên GPU T4 miễn phí của Colab.

In [16]:
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.001300,0.001239,0.011713,1.000000,156672.000000
2,0.000700,0.000706,0.007180,1.000000,313344.000000
3,0.000700,0.000696,0.007097,1.000000,470016.000000


TrainOutput(global_step=117, training_loss=0.1371285950876454, metrics={'train_runtime': 2730.4417, 'train_samples_per_second': 0.336, 'train_steps_per_second': 0.043, 'total_flos': 3747358725636096.0, 'train_loss': 0.1371285950876454, 'epoch': 3.0})

In [17]:
import torch
import transformers
import trl
import inspect
import trl.kernels.chunked_logprob as clp

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("Kernel file:", inspect.getfile(clp))
print("_MM_OUT_DTYPE:", getattr(clp, "_MM_OUT_DTYPE", "không tìm thấy"))

PyTorch: 2.11.0+cu130
Transformers: 4.57.6
TRL: 1.16.0
Kernel file: /usr/local/lib/python3.13/dist-packages/trl/kernels/chunked_logprob.py
_MM_OUT_DTYPE: False


In [18]:
save_path = '/content/drive/MyDrive/Project/Xet_Khen_Thuong/Models/Model-Qwen-Ner-Rel'

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Đã lưu model tại:", save_path)


Đã lưu model tại: /content/drive/MyDrive/Project/Xet_Khen_Thuong/Models/Model-Qwen-Ner-Rel


## Kiểm tra model sau fine-tuning

In [19]:
# model.eval()


# def build_prompt(question: str) -> str:
#     messages = [
#         {"role": "system", "content": SYSTEM_PROMPT},
#         {"role": "user", "content": question},
#     ]
#     return tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True
#     )


# question = "Giải thích LoRA bằng ngôn ngữ đơn giản."

# prompt = build_prompt(question)

# inputs = tokenizer(
#     prompt,
#     return_tensors="pt"
# ).to(model.device)

# with torch.no_grad():
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=200,
#         temperature=0.7,
#         top_p=0.9,
#         do_sample=True,
#         repetition_penalty=1.1,
#         pad_token_id=tokenizer.pad_token_id,
#     )

# answer = tokenizer.decode(
#     outputs[0][inputs["input_ids"].shape[1]:],
#     skip_special_tokens=True
# )

# print("Câu hỏi:")
# print(question)

# print("\nTrả lời:")
# print(answer)


In [20]:
# def chat(question: str) -> str:
#     prompt = build_prompt(question)

#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt"
#     ).to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=256,
#             temperature=0.7,
#             top_p=0.9,
#             do_sample=True,
#             repetition_penalty=1.1,
#             pad_token_id=tokenizer.pad_token_id,
#         )

#     response = tokenizer.decode(
#         outputs[0][inputs["input_ids"].shape[1]:],
#         skip_special_tokens=True
#     )

#     return response


In [21]:
# print(
#     chat(
#         "Tại sao Transformer phù hợp với xử lý ngôn ngữ tự nhiên?"
#     )
# )


#XÀI LOCAL LLM CHO RAG

In [22]:
# USE_LOCAL_MODEL = True  # đổi thành False nếu muốn dùng Gemini API thay vì model cục bộ

In [23]:
# if USE_LOCAL_MODEL:
#     import os
#     import torch
#     from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
#     from peft import PeftModel

#     LOCAL_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
#     LOCAL_ADAPTER_PATH = "qwen-vietnamese-lora-final"  # đổi nếu bạn đặt tên thư mục khác

#     # Dùng bfloat16 xuyên suốt để tránh lỗi GradScaler/BFloat16 thường gặp khi
#     # trộn lẫn fp16 và bf16 (lưu ý: đây là suy luận/inference nên không cần
#     # GradScaler, nhưng vẫn nên đồng nhất dtype để tránh cảnh báo/lỗi không cần thiết).
#     compute_dtype = torch.bfloat16

#     bnb_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_quant_type="nf4",
#         bnb_4bit_use_double_quant=True,
#         bnb_4bit_compute_dtype=compute_dtype,
#     )

#     local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME, trust_remote_code=True)
#     if local_tokenizer.pad_token is None:
#         local_tokenizer.pad_token = local_tokenizer.eos_token

#     print(f"[Local LLM] Đang tải model gốc: {LOCAL_MODEL_NAME} ...")
#     local_base_model = AutoModelForCausalLM.from_pretrained(
#         LOCAL_MODEL_NAME,
#         quantization_config=bnb_config,
#         torch_dtype=compute_dtype,
#         device_map="auto",
#         trust_remote_code=True,
#     )
#     # Đây là suy luận (inference), không phải training, nên bật lại cache để sinh
#     # văn bản nhanh hơn (khác với lúc fine-tune phải tắt use_cache).
#     local_base_model.config.use_cache = True

#     if os.path.isdir(LOCAL_ADAPTER_PATH):
#         print(f"[Local LLM] Tìm thấy adapter LoRA tại '{LOCAL_ADAPTER_PATH}', đang gắn vào model gốc...")
#         local_model = PeftModel.from_pretrained(local_base_model, LOCAL_ADAPTER_PATH)
#     else:
#         print(
#             f"[Local LLM] Không thấy thư mục adapter '{LOCAL_ADAPTER_PATH}'. "
#             "Dùng thẳng model Qwen2.5-1.5B-Instruct gốc (khuyến nghị cho RAG)."
#         )
#         local_model = local_base_model

#     local_model.eval()
#     print("[Local LLM] Sẵn sàng.")
# else:
#     local_model = None
#     local_tokenizer = None


In [24]:
# import torch

# LOCAL_SYSTEM_PROMPT = (
#     "Bạn là trợ lý AI sử dụng tiếng Việt. "
#     "Hãy trả lời chính xác, rõ ràng và dễ hiểu."
# )


# def local_generate(prompt, max_new_tokens=400, temperature=0.3):
#     """Sinh câu trả lời bằng model Qwen2.5 cục bộ (dùng khi USE_LOCAL_MODEL=True)."""
#     messages = [
#         {"role": "system", "content": LOCAL_SYSTEM_PROMPT},
#         {"role": "user", "content": prompt},
#     ]
#     text = local_tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     inputs = local_tokenizer(text, return_tensors="pt").to(local_model.device)

#     with torch.no_grad():
#         outputs = local_model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             temperature=temperature,
#             top_p=0.9,
#             do_sample=True,
#             repetition_penalty=1.1,
#             pad_token_id=local_tokenizer.pad_token_id,
#         )

#     return local_tokenizer.decode(
#         outputs[0][inputs["input_ids"].shape[1]:],
#         skip_special_tokens=True,
#     )


# print("[Local LLM] Đã định nghĩa hàm local_generate().")

In [25]:
# import os
# from google.colab import files

# os.makedirs("documents", exist_ok=True)
# uploaded = files.upload()

# for fname in uploaded.keys():
#     dest = os.path.join("documents", fname)
#     os.rename(fname, dest)

# print("\nĐã tải lên và lưu vào thư mục 'documents/':")
# for fname in uploaded.keys():
#     print(" -", fname)

In [26]:
# !pip install -q sentence-transformers faiss-cpu

In [27]:
# print(local_generate("Xin chào. Hãy trả lời một câu ngắn bằng tiếng Việt."))

In [28]:
# import glob
# import numpy as np
# from sentence_transformers import SentenceTransformer
# import faiss

# try:
#     from pypdf import PdfReader
# except ImportError:
#     PdfReader = None

# # ---------------- Cấu hình ----------------
# # Model embedding chạy CỤC BỘ, MIỄN PHÍ (đa ngôn ngữ, hỗ trợ tốt tiếng Việt).
# EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# DOCUMENTS_FOLDER = "/content/documents"
# CHUNK_SIZE = 500      # số từ mỗi đoạn
# CHUNK_OVERLAP = 80    # số từ chồng lấn giữa 2 đoạn liên tiếp
# TOP_K = 3             # số đoạn liên quan nhất lấy ra để hỏi LLM


# # ---------------- Đọc dữ liệu ----------------
# def read_txt_file(path):
#     with open(path, "r", encoding="utf-8", errors="ignore") as f:
#         return f.read()

# def read_pdf_file(path):
#     if PdfReader is None:
#         raise RuntimeError("Chưa cài pypdf. Chạy: !pip install pypdf")
#     reader = PdfReader(path)
#     return "\n".join(page.extract_text() or "" for page in reader.pages)

# def load_documents(folder):
#     docs = []
#     paths = glob.glob(os.path.join(folder, "**", "*.*"), recursive=True)
#     for path in paths:
#         ext = os.path.splitext(path)[1].lower()
#         try:
#             if ext in (".txt", ".md"):
#                 text = read_txt_file(path)
#             elif ext == ".pdf":
#                 text = read_pdf_file(path)
#             else:
#                 continue
#             if text.strip():
#                 docs.append({"source": os.path.basename(path), "text": text})
#         except Exception as e:
#             print(f"[Cảnh báo] Không đọc được file {path}: {e}")
#     return docs


# # ---------------- Chia đoạn (chunking) ----------------
# def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
#     words = text.split()
#     if not words:
#         return []
#     chunks = []
#     step = max(chunk_size - overlap, 1)
#     for start in range(0, len(words), step):
#         chunk_words = words[start:start + chunk_size]
#         if chunk_words:
#             chunks.append(" ".join(chunk_words))
#         if start + chunk_size >= len(words):
#             break
#     return chunks


# # ---------------- Pipeline RAG ----------------
# class RAGPipeline:
#     def __init__(self, embedding_model_name=EMBEDDING_MODEL_NAME):
#         print(f"[Khởi tạo] Đang tải model embedding cục bộ: {embedding_model_name} ...")
#         self.embedder = SentenceTransformer(embedding_model_name)
#         self.index = None
#         self.chunk_texts = []
#         self.chunk_sources = []

#     def build_index(self, folder):
#         docs = load_documents(folder)
#         if not docs:
#             raise RuntimeError(
#                 f"Không tìm thấy file .txt/.pdf nào trong thư mục '{folder}'. "
#                 f"Hãy chạy lại Bước 3 để tải tài liệu lên."
#             )

#         print(f"[Nạp dữ liệu] Đọc được {len(docs)} tài liệu. Đang chia đoạn (chunking)...")
#         for doc in docs:
#             for chunk in chunk_text(doc["text"]):
#                 self.chunk_texts.append(chunk)
#                 self.chunk_sources.append(doc["source"])

#         print(f"[Nạp dữ liệu] Tổng cộng {len(self.chunk_texts)} đoạn (chunks).")
#         print("[Embedding] Đang tính vector cho từng đoạn (chạy cục bộ, miễn phí)...")

#         embeddings = self.embedder.encode(
#             self.chunk_texts,
#             batch_size=32,
#             show_progress_bar=True,
#             normalize_embeddings=True,
#         )
#         embeddings = np.asarray(embeddings, dtype="float32")

#         dim = embeddings.shape[1]
#         self.index = faiss.IndexFlatIP(dim)
#         self.index.add(embeddings)

#         print("[Hoàn tất] Chỉ mục FAISS đã sẵn sàng.")

#     def retrieve(self, query, top_k=TOP_K):
#         query_vec = self.embedder.encode([query], normalize_embeddings=True)
#         query_vec = np.asarray(query_vec, dtype="float32")
#         scores, indices = self.index.search(query_vec, top_k)
#         results = []
#         for score, idx in zip(scores[0], indices[0]):
#             if idx == -1:
#                 continue
#             results.append({
#                 "text": self.chunk_texts[idx],
#                 "source": self.chunk_sources[idx],
#                 "score": float(score),
#             })
#         return results

#     def generate_answer(self, query, contexts):
#         context_block = "\n\n".join(
#             f"[Nguon: {c['source']}]\n{c['text']}" for c in contexts
#         )
#         prompt = f"""Bạn là một trợ lý trả lời câu hỏi dựa HOÀN TOÀN trên các đoạn
# văn bản ngữ cảnh được cung cấp bên dưới. Nếu ngữ cảnh không chứa đủ thông tin
# để trả lời, hãy nói rõ là bạn không tìm thấy thông tin đó trong tài liệu,
# KHÔNG được tự bịa ra thông tin.

# ### NGỮ CẢNH:
# {context_block}

# ### CÂU HỎI:
# {query}

# ### TRẢ LỜI (ngắn gọn, chính xác, dựa trên ngữ cảnh trên):"""

#         # Model 1.5B khá nhỏ nên giữ prompt ngắn gọn, temperature thấp để bám
#         # sát ngữ cảnh, hạn chế bịa thông tin.
#         return local_generate(prompt, max_new_tokens=400, temperature=0.3)

#     def ask(self, query, top_k=TOP_K):
#         contexts = self.retrieve(query, top_k=top_k)
#         if not contexts:
#             return "Không tìm thấy đoạn văn bản nào liên quan trong tài liệu.", []
#         answer = self.generate_answer(query, contexts)
#         return answer, contexts

# print("Đã định nghĩa xong pipeline RAGPipeline.")


In [29]:
# rag = RAGPipeline()
# rag.build_index(DOCUMENTS_FOLDER)

In [30]:
# query = "What is AIM-MD?"

# answer, contexts = rag.ask(query)

# print("--- TRẢ LỜI ---")
# print(answer)

# print("\n--- NGUỒN THAM KHẢO (retrieval) ---")
# for i, c in enumerate(contexts, 1):
#     preview = c["text"][:150].replace("\n", " ")
#     print(f"[{i}] {c['source']} (độ tương đồng={c['score']:.3f}): {preview}...")
